# Week 06: Finding useful information — the case

**Start here:** A learner asks “Can I renew my book?” Policy P2 says “library loans may be extended once.” Which passage should search find?

**Your goal:** Compare search strategies while preserving source and access information.

Run the prepared cells from top to bottom. Predict the result before changing one input. These small demonstrations isolate an idea; they are not trained production models.

**Concepts to point to:** Chunk, Keyword search, Vector search, Access filter.

# Week 06: Finding useful information

**Goal:** Compare search strategies while preserving source and access information.

This notebook runs with the Python standard library. All records are fictional. Run cells top to bottom, then change one input and explain the observed result. Classroom stand-ins are labelled; they are not trained generative models.

## 1. Set up the case

Five fictional college policies must answer student questions.

In [ ]:
import re
policies = [{'id': 'P1', 'text': 'Student travel costs, including bus tickets, are not reimbursed.', 'role': 'student', 'version': '2027-01'}, {'id': 'P2', 'text': 'Library loans may be extended once for seven days.', 'role': 'student', 'version': '2027-01'}, {'id': 'P3', 'text': 'Students may request one deadline extension within seven days of the original due date.', 'role': 'student', 'version': '2027-01'}, {'id': 'P4', 'text': 'Academic appeals must be filed within ten days of the result.', 'role': 'student', 'version': '2027-01'}, {'id': 'P5', 'text': 'Staff payroll changes require manager approval before any record is changed.', 'role': 'staff', 'version': '2027-01'}]
synonyms={'renew':'extended','book':'loans','refund':'reimbursed','challenge':'appeals','salary':'payroll'}
def tokens(s): return set(re.findall(r'[a-z]+',s.lower()))
def search(q,role='student',k=2,expand=False):
    terms=tokens(q)
    if expand: terms|={synonyms[t] for t in terms if t in synonyms}
    allowed=[p for p in policies if p['role']==role or p['role']=='student']
    ranked=sorted(allowed,key=lambda p: len(terms & tokens(p['text'])),reverse=True)
    return [p for p in ranked if terms & tokens(p['text'])][:k]
cases=[('renew a book','P2'),('refund for bus','P1'),('deadline extension','P3'),('appeal deadline','P4'),('two extensions','P3'),('library loans','P2'),('travel costs','P1'),('file an appeal','P4'),('book extension','P2'),('challenge result','P4')]
for expand in (False,True):
    hits=[target in [p['id'] for p in search(q,expand=expand)] for q,target in cases]
    print('synonym proxy',expand,'recall@2:',sum(hits),'/',len(hits),'misses:',[q for (q,_),hit in zip(cases,hits) if not hit])
assert not search('salary',role='student')


Token overlap and a handwritten synonym map are retrieval proxies, not actual learned embeddings. Check which role is allowed to see P5.

## 3. Record your evidence

| Original setting | Your one change | Observed output | Explanation | Limitation |
|---|---|---|---|---|
| | | | | |

**Exit question:** The synonym mapping is a production embedding model. Explain why this is wrong.

**Reading:** [Microsoft, Retrieval augmented generation](https://learn.microsoft.com/en-us/azure/foundry/concepts/retrieval-augmented-generation), [Lewis et al., Retrieval-Augmented Generation](https://arxiv.org/abs/2005.11401)

## How a numerical representation changes search
These manually supplied vectors illustrate cosine similarity. They are not learned embeddings or evidence of a model benchmark.

In [ ]:
import math
query_vector=[1,0,0]
prepared_vectors={'P1':[0,1,0],'P2':[.95,.05,0],'P3':[.5,.4,.1]}
def cosine(a,b):
    denom=math.sqrt(sum(x*x for x in a))*math.sqrt(sum(x*x for x in b))
    return sum(x*y for x,y in zip(a,b))/denom if denom else 0
for pid,vector in sorted(prepared_vectors.items(),key=lambda item:cosine(query_vector,item[1]),reverse=True):
    print(pid,round(cosine(query_vector,vector),3))
print('A learned embedding model would produce the vectors. Check relevance and permissions separately.')


## Explain and transfer

1. What did you change, and what happened? Remove a synonym or lower top-k from 2 to 1 and compare recall.
2. Which diagram block produced the difference?
3. What is one limitation of the demonstration?
4. Fresh situation: A staff-only policy has a perfect keyword match. Should the student see it?

**Evidence to save:** Corpus manifest and retrieval misses with evidence IDs.